In [1]:
# This allows us to import from all folders one level up from notebooks folder - run 1 time
import sys
from pathlib import Path

print('All paths pre-append')
for i,p in enumerate(sys.path):
    print(f"{i}: {p}")
print('-'*100)

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
print('All paths post-append')
for i,p in enumerate(sys.path):
    print(f"{i}: {p}")


All paths pre-append
0: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python311.zip
1: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11
2: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/lib-dynload
3: 
4: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/site-packages
----------------------------------------------------------------------------------------------------
All paths post-append
0: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python311.zip
1: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11
2: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/lib-dynload
3: 
4: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/site-packages
5: /Users/irabandutta/Developer/2026-08-llm-from-scratch


# Imports

In [2]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from src.model.llm_config import LLMConfig
from src.model.llm import LLM
from src.training.trainer import LLMTrainerConfig, LLMTrainer, TokenBatchLoader

# Define Model and Train Config and train for a few steps

In [3]:
# ======== SET Global Configs ========
B = 8
T = 128
STEPS = 100
DEVICE = 'auto'

# ======== SET SEED ========
seed=42
torch.manual_seed(seed)


# ======== DEFINE Model Config ========
llm_config = LLMConfig(
    vocab_size=50304,     # Divisible by 128
    ctx_len=T,
    d_model=128, 
    n_layer=4,
    ff_ratio=4,
    dropout=0.0,
    eps=1e-5,
    bias=False,
    position_embedding='sinusoidal',
    rotary_embedding=False,
    attention='gqa',
    normalization='layernorm',
    n_heads=8, 
    n_groups=4,
    use_flash=False, 
    attn_debug=False
)
print(llm_config)
print('-'*50)


# ======== DEFINE Trainer Config ========
trainer_config = LLMTrainerConfig(
    num_steps=STEPS,
    batch_size=B,
    learning_rate=3e-4,
    weight_decay=0.01,
    beta1=0.9,
    beta2=0.95,
    use_lr_scheduler=False,
    warmup_steps=15,
    min_lr=0.1*3e-4,
    grad_clip=1.0,
    log_interval=10,
    eval_interval=50,
    eval_steps=16,
    to_save_checkpoint=False,
    checkpoint_interval=1000,
    device=DEVICE
)
print(trainer_config)
print('-'*50)

# ======== Instantiate Train Batch Loaders ========
print(f'Create generator for loading train data of shape ({B}, {T})')
binary_file_path = '../data/tinystories/processed/train.bin'
tok_bl = TokenBatchLoader(
    B=trainer_config.batch_size, 
    T=llm_config.ctx_len, 
    binary_file_path=binary_file_path, 
    dtype=np.uint16, 
    debug=False
)

# ======== Instantiate Val Batch Loaders ========
print(f'Create generator for loading validation data of shape ({B}, {T})')
binary_file_path = '../data/tinystories/processed/val.bin'
val_token_len = len(np.memmap(filename=binary_file_path, dtype=np.uint16, mode='r'))
tok_bl_val = TokenBatchLoader(
    B=trainer_config.batch_size, 
    T=llm_config.ctx_len, 
    binary_file_path=binary_file_path, 
    dtype=np.uint16, 
    debug=False
)

# ======== Instantiate model ========
model = LLM(config=llm_config)

# ======== Instantiate trainer ========
trainer = LLMTrainer(
    config=trainer_config,
    model=model,
    train_loader=tok_bl,
    val_loader=tok_bl_val
)

# ======== Start Training ========
print('Starting Training...')
print('-'*50)
trainer.train()

LLMConfig(vocab_size=50304, ctx_len=128, d_model=128, n_layer=4, ff_ratio=4, dropout=0.0, eps=1e-05, bias=False, position_embedding='sinusoidal', rotary_embedding=False, attention='gqa', normalization='layernorm', n_heads=8, n_groups=4, use_flash=False, attn_debug=False)
--------------------------------------------------
LLMTrainerConfig(num_steps=100, batch_size=8, learning_rate=0.0003, weight_decay=0.01, beta1=0.9, beta2=0.95, use_lr_scheduler=False, warmup_steps=15, min_lr=2.9999999999999997e-05, grad_clip=1.0, log_interval=10, eval_interval=50, eval_steps=16, to_save_checkpoint=False, checkpoint_interval=1000, checkpoint_dir='./checkpoints/2026_08_22_15_06_56', device='auto')
--------------------------------------------------
Create generator for loading train data of shape (8, 128)
Loaded 471.872517M tokens
1 epoch ~ 460813 steps
--------------------------------------------------
Create generator for loading validation data of shape (8, 128)
Loaded 4.743928M tokens
1 epoch ~ 4632 

# Save Checkpoint

In [4]:
# What to save in ckpt
# Model stae_dict()
# Optim stae_dict()
# Configs: model & trainer
# Imp states/attributes of TrainerClass: curr_step, loss_hist (train and valid)
# Random number state
# Misc: curr_idx of train data loader object (so that we can resume from the exact location when we stopped training)


# Create a dict of key value pairs and use torch.save() to dump the ckpt

In [5]:
sample_ckpt = {
    'model_state_dict': model.state_dict(),
    'optim_state_dict': trainer.optimizer.state_dict(),
    'model_config': model.config,
    'trainer_config': trainer.config,
    'step': trainer.step,
    'train_loss_hist': trainer.train_loss_hist,
    'val_loss_hist': trainer.val_loss_hist,
    'rng_state': torch.get_rng_state(),
    'train_loader_curr_idx': {
        'curr_idx': trainer.train_loader.curr_idx
    }
}

In [6]:
# Saves the ckpt into the file whose path is given
torch.save(sample_ckpt, './sample_ckpt.pt')

# Load Checkpoint

In [7]:
sample_ckpt_path = './sample_ckpt.pt'
loaded_samople_ckpt = torch.load(sample_ckpt_path, map_location=trainer.device.type)

/var/folders/y2/hgp77z211t5_wgb6zw7l23hc0000gn/T/ipykernel_20791/2710410728.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_samople_ckpt = torch.load(sample_ckpt_